<div align="center">
  <a href="https://colab.research.google.com/github/PrunaAI/ai-efficiency-courses/blob/main/solutions/05-benchmark_llm_bits.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

---
**💡 Tip**: Click the button above to open this notebook in Google Colab for free GPU access!

## Installation

This notebook includes automatic setup cells that will install the project from git repository with UV.

**Note**: Run the setup cells below before starting the exercises.

In [ ]:
# Install project directly from git repository
!uv pip install git+https://github.com/PrunaAI/ai-efficiency-courses.git

# 05: Benchmark LLM bit precision

## 1. Imports

In [ ]:
import gc
import copy
# import os
# CACHE_PATH = '<path_to_cache>'
# os.environ["TORCH_HOME"] = CACHE_PATH
# os.environ["HF_HOME"] = CACHE_PATH
# os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
# os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH
# os.environ["TRANSFORMERS_CACHE"] = CACHE_PATH

import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
# from huggingface_hub import login; login(token="<hf_token>")

from pruna_pro import SmashConfig
from pruna_pro import smash
from pruna.data.pruna_datamodule import PrunaDataModule
from pruna.evaluation.evaluation_agent import EvaluationAgent
from pruna.evaluation.metrics import (
    DiskMemoryMetric,
    EnergyConsumedMetric,
    TotalMACsMetric,
    TotalParamsMetric,
    TotalTimeMetric,
    TorchMetricWrapper,
)
from pruna.evaluation.task import Task

Multiple distributions found for package optimum. Picked distribution: optimum


## 2. Utils

The utils functions help for:
- Load from a list of (small) models.  eel free to try other models until the GPU memory is not enough!
- Make plots.
- Iterate over evaluation and model configurations.

In [ ]:
model_ids = [
    # "facebook/opt-125m",
    # "facebook/opt-350m",
    # "facebook/opt-1.3b",
    # "facebook/opt-2.7b",
    # "meta-llama/Llama-3.2-1B",
    # "meta-llama/Llama-3.2-1B-Instruct",
    # "meta-llama/Llama-3.2-3B-Instruct",
    # "google/gemma-3-1b-it",
    # "google/gemma-3-4b-it",
    # "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    # "microsoft/Phi-4-mini-instruct",
    # "HuggingFaceTB/SmolLM-135M",
    # "HuggingFaceTB/SmolLM-135M-instruct",
    # "HuggingFaceTB/SmolLM-360M",
    # "HuggingFaceTB/SmolLM-360M-Instruct",
    # "HuggingFaceTB/SmolLM-1.7B",
    # "HuggingFaceTB/SmolLM-1.7B-Instruct",
    # "HuggingFaceTB/SmolLM2-135M",
    # "HuggingFaceTB/SmolLM2-135M-Instruct",
    # "HuggingFaceTB/SmolLM2-360M",
    # "HuggingFaceTB/SmolLM2-360M-Instruct",
    # "HuggingFaceTB/SmolLM2-1.7B",
    # "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "PleIAs/Pleias-350m-Preview",
    # "PleIAs/Pleias-Pico",
    # "PleIAs/Pleias-1.2b-Preview",
    # "PleIAs/Pleias-Nano",
    # "PleIAs/Pleias-3b-Preview",
]

# Load model and tokenizer for   first model in list
model_id = model_ids[0]
model = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

## 2. Evaluation function for all metrics

In [ ]:
def evaluate_model(model, tokenizer, dataset="WikiText"):
    """Evaluate a language model using multiple metrics.

    Args:
        model: The language model to evaluate (AutoModelForCausalLM)
        tokenizer: The tokenizer associated with the model (AutoTokenizer)
        dataset: Name of the dataset to evaluate on (default: "WikiText")

    Returns:
        dict: Dictionary containing evaluation results with metrics like:
            - Elapsed time (inference speed)
            - GPU memory usage
            - Energy consumption
            - Model architecture details
            - Perplexity score
    """
    metrics = [
        TotalTimeMetric(
            n_iterations=100,
            n_warmup_iterations=10,
            device="cuda",
            timing_type="sync",
        ),
        DiskMemoryMetric(),
        EnergyConsumedMetric(n_iterations=100, n_warmup_iterations=10, device="cuda"),
        TotalMACsMetric(),
        TotalParamsMetric(),
        TorchMetricWrapper(metric_name="perplexity", call_type="y_gt"),
    ]

    task = Task(
        metrics, datamodule=PrunaDataModule.from_string(dataset, tokenizer=tokenizer)
    )
    eval_agent = EvaluationAgent(task)

    # Run evaluation
    return eval_agent.evaluate(model)

In [4]:
def plot_evaluation_results(evaluation_results):
    """
    Plot evaluation metrics for different model configurations.

    Args:
        evaluation_results (dict): Dictionary containing evaluation metrics for each model configuration
    """
    # Get metrics from first result (assuming all configs have same metrics)
    first_result = next(iter(evaluation_results.values()))
    metrics = [
        key
        for key in first_result.keys()
        if isinstance(first_result[key], (int, float))
    ]
    data = {}

    for metric in metrics:
        data[metric] = [
            evaluation_results[model][metric] for model in evaluation_results
        ]

    # Calculate number of rows and columns for subplots
    n_metrics = len(metrics)
    n_cols = 3  # Display 3 plots per row
    n_rows = (
        n_metrics + n_cols - 1
    ) // n_cols  # Ceiling division to get number of rows needed

    # Create figure with subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    fig.suptitle("Comparison of Models with Different Bit Precision", y=1.02)

    # Flatten axes array to make iteration easier
    axes = axes.flatten()

    # Plot each metric
    for idx, (metric, values) in enumerate(data.items()):
        ax = axes[idx]
        ax.bar(list(evaluation_results.keys()), values)
        ax.set_ylabel(metric)
        ax.tick_params(axis="x", rotation=45)

    # Hide empty subplots if any
    for idx in range(len(metrics), len(axes)):
        axes[idx].set_visible(False)

    plt.tight_layout()
    plt.show()

## 3. Benchmark LLMs bit precision

We recommend to checkout the [Pruna documentation](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/evaluation.html) for access to AI efficiency functions.


### 3.1 Evaluate quantization configurations

**Implementation task:**
- Apply all the quantization configurations on the base model.
- Evaluate all the quantization configurations on GPU by using the `evaluate_model` function.

In [ ]:
def evaluate_configs(model, smash_configs, dataset="WikiText"):
    """Evaluate multiple quantization configurations on a model.

    Args:
        model: The base model to evaluate quantization on (AutoModelForCausalLM)
        smash_configs: List of quantization configurations to evaluate
        dataset: Name of the dataset to evaluate on (default: "WikiText")

    Returns:
        dict: Dictionary mapping quantization bit width to evaluation results, where each result contains:
            - Elapsed time
            - GPU memory usage
            - Energy consumption
            - Model architecture details
            - Perplexity score
            Returns None for configurations that fail
    """
    results = {}

    ### To Complete ###

### 3.2 Evaluate different bit precision for LLM.int8()

**Implementation task:**
- Define LLM.int8() quantization configurations for all bits that are available.
- Quantize and evalute the base model with all the quantization configurations.
- Plot the results.

**Questions:**
- Which bit precision is the best/worst in terms of quality?
- What is the impact of bit precision on the speed?
- What is the impact of bit precision on the memory?

In [ ]:
smash_configs = []

### To Complete ###

### 3.2 Evaluate different bit precision for Quanto

**Implementation task:**
- Define Quanto quantization configurations for all bits that are available.
- Quantize and evalute the base model with all the quantization configurations.
- Plot the results.

**Questions:**
- Which bit precision is the best/worst in terms of quality?
- What is better between qfloat8 and qint8?
- What is the impact of bit precision on the speed?
- What is the impact of bit precision on the memory?

In [ ]:
# Create list of SmashConfig with different quantization methods
smash_configs = []

### To Complete ###

### 3.3 Evaluate different bit precision for HQQ

**Implementation task:**
- Define HQQ quantization configurations for all bits that are available.
- Quantize and evalute the base model with all the quantization configurations.
- Plot the results.

**Questions:**
- Which bit precision is the best/worst in terms of quality?
- What is the impact of bit precision on the speed?
- What is the impact of bit precision on the memory?
- What would you recommend among all quantization configs that you explored?

In [ ]:
# Create list of SmashConfig with different quantization methods
smash_configs = []

### To Complete ###